# Rich Chess Ebook — extraction pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/loloof64/RichChessEbooks/blob/main/notebooks/rce_pipeline.ipynb)

Opening it from that badge loads the notebook as it stands on `main`, and the
cells below clone the pipeline code from the same place — so a pushed commit is
one reload away, with nothing to upload. Edits made here belong to the session
only; `File → Save a copy in Drive` is what forks it away from the repository.

Turns **PDF chess books** into `.rce` archives the Flutter app can read, several in
one pass so their numbers can be read side by side.

Two notations are read straight from the text layer: figurine Unicode, and plain
letters in `en`, `fr`, `de`, `es`, `it` or `nl`.

A book whose piece symbols are **drawn** — a scan, or a figurine font — holds no
readable symbol in that layer at all. Step 4 reports those as
`needs_glyph_recovery = True`, and a trained classifier reads their symbols off the
page images and writes them back in as figurines.

**The classifier is handed to every book, once, in step 3.** `run()` engages it only
on the books whose detection asked for it, so there is no per-book flag to set and no
way to lose a scanned book by forgetting one — which is the mistake this notebook was
rewritten to make impossible.

The logic is not in these cells but in the repository's `rce_pipeline` package: a
notebook is neither testable nor readable as a diff, whereas each step here is a
module that can be fixed and re-run on its own.

| Step | Module | Artefact written |
| --- | --- | --- |
| 1. Text + per-character geometry | `extract.py` | `work/<book>/01_pages.json` |
| 1c. Piece symbols read off the image | `scan.py`, `glyphs.py` | `work/<book>/01b_glyphs.json` |
| 2. Notation detection | `notation.py` | `work/<book>/02_notation.json` |
| 3a. Tokenising | `tokenize.py` | `work/<book>/03_tokens.json` |
| 3b + 4. Move tree, legality, FEN | `parse.py` | `work/<book>/04_moves.json` |
| 5. Packaging | `package.py` | `<book>.rce` |

Each step reads the previous one's artefact, so editing `parse.py` and re-running does
not redo extraction — by far the slowest part.


In [1]:
!rm -rf RichChessEbooks

## 1. Install


In [2]:
!pip install -q pymupdf chess
# Only needed for a scanned or figurine-font book (step 4b):
!pip install -q scikit-learn scikit-image pillow


## 2. Get the pipeline code

Set `REPO_URL` to clone from GitHub. Leave it at `None` and the cell asks you to
upload a ZIP of the `pipeline/` directory instead.


In [3]:
REPO_URL = "https://github.com/loloof64/RichChessEbooks.git"  # or None to upload a ZIP

import os, sys, zipfile

if REPO_URL:
    if not os.path.isdir("RichChessEbooks"):
        !git clone --depth 1 $REPO_URL
    PIPELINE_DIR = "RichChessEbooks/pipeline"
elif os.path.isdir("pipeline"):
    PIPELINE_DIR = "pipeline"
else:
    from google.colab import files
    print("Upload a ZIP containing the pipeline/ directory")
    for name in files.upload():
        with zipfile.ZipFile(name) as archive:
            archive.extractall(".")
    PIPELINE_DIR = "pipeline"

sys.path.insert(0, os.path.abspath(PIPELINE_DIR))

import rce_pipeline
from rce_pipeline import extract, notation, tokenize, parse, package, pipeline
print("rce_pipeline", rce_pipeline.__version__, "from", PIPELINE_DIR)

# Which code actually got loaded. `rm -rf` above removes the clone, but it
# cannot evict a module already imported into a live kernel: without a restart
# Python keeps the old one and the run silently repeats itself, down to the
# last digit. These two lines make that visible instead.
print("loaded from :", os.path.dirname(parse.__file__))
if REPO_URL and os.path.isdir("RichChessEbooks"):
    !git -C RichChessEbooks log --oneline -1
if os.path.abspath(PIPELINE_DIR) not in os.path.abspath(parse.__file__):
    print("\n/!\\ STALE: the loaded package is not the one just cloned.")
    print("    Runtime -> Restart session, then run all.")


Cloning into 'RichChessEbooks'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 227 (delta 19), reused 220 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (227/227), 374.44 KiB | 4.11 MiB/s, done.
Resolving deltas: 100% (19/19), done.
rce_pipeline 0.1.0 from RichChessEbooks/pipeline


## 3. The books, and the classifier

`BOOKS` maps a short label to a PDF path — the label is what names the reports and
the `.rce` files below. Reading from Drive is worth it for anything large; leave the
dict empty to upload instead.

`GLYPH_MODEL` is the trained piece classifier, and it is set **once, here, for all
the books**. Passing it to a book that does not need it costs nothing: `run()` checks
each book's own `needs_glyph_recovery` before engaging it.

**Uploads happen once.** Whatever you upload is written into `/content/books` and
`/content/model` and picked up again on every later run of this cell, so re-running it
after an error does not ask for the 48 MB classifier a second time. Set `REUPLOAD =
True` to replace what is there. Files already sitting in `/content` — from an earlier
upload in this session — are adopted rather than asked for again.

For anything large, mounting Drive still beats uploading:

```python
from google.colab import drive; drive.mount("/content/drive")
```

then point `BOOKS` and `GLYPH_MODEL` straight at `/content/drive/MyDrive/...`.


In [ ]:
from google.colab import drive; drive.mount("/content/drive")

# label -> path. Leave empty to upload.
BOOKS = {
    "SuperAttaquant": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/CommentDevenirSuperAttaquant.pdf",
    "BoussoleSurEchiquier": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/BoussoleSurEchiquier.pdf",
    "ChessStrategy_Grivas_1": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/ChessTrategy_Grivas_1.pdf",
    "Tactics": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_texte/TacticsForTournamentPlayer.pdf",
    # Complete books, added to widen a corpus measured almost entirely on one
    # of the four above. Copy them to Drive under these names first; a book
    # that is not there is skipped with a warning rather than stopping the run.
    "Sakaev": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/PositionalChess_Sakaev_1.pdf",
    "Markos": "/content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/UnderTheSurface_Markos.pdf",
}

# label -> (first page, last page). A complete book is not a fixture: most of
# its pages are prose, front matter or index, and a range has to be chosen for
# each. `pipeline/scripts/choose_pages.py` is what chooses one — it ranks by
# pages opening a game from the initial position, not by notation density,
# because the densest pages are the analysis fragments that cannot be replayed
# at all. Books absent from this map are read over DEFAULT_PAGES.
PAGES = {
    "Sakaev": (38, 49),   # 722 move numbers, 8 game starts
    "Markos": (85, 96),   # 240 move numbers, 4 game starts
}
DEFAULT_PAGES = (1, 40)


def pages_of(label):
    return PAGES.get(label, DEFAULT_PAGES)

GLYPH_MODEL = "/content/drive/MyDrive/entrainement_ocr_echecs/chess_glyphs_classifier.zip"

BOOKS_DIR = "/content/books"
MODEL_DIR = "/content/model"
REUPLOAD = False  # True to ignore what is already there and upload again

import glob, os, re


def _save(target_dir):
    """Prompt for files and write them into `target_dir` under their own names.

    `files.upload()` hands back `{name: bytes}`, so the bytes are written here
    rather than left where Colab dropped them. Colab saves a second upload of
    the same book as `book (1).pdf`, which would then be read as a second book.
    """
    from google.colab import files

    os.makedirs(target_dir, exist_ok=True)
    for name, data in files.upload().items():
        with open(os.path.join(target_dir, name), "wb") as handle:
            handle.write(data)


def _discover(*directories, pattern):
    """Files matching `pattern`, first directory winning, `name (1)` folded away.

    Sorted shortest basename first so that `book.pdf` beats the `book (1).pdf`
    Colab wrote beside it: plain alphabetical order puts the duplicate first,
    the space sorting before the dot.
    """
    found = {}
    for directory in directories:
        paths = glob.glob(os.path.join(directory, pattern))
        for path in sorted(paths, key=lambda p: (len(os.path.basename(p)), p)):
            stem = os.path.splitext(os.path.basename(path))[0]
            found.setdefault(re.sub(r" \(\d+\)$", "", stem), path)
    return found


if not BOOKS:
    if REUPLOAD or not _discover(BOOKS_DIR, "/content", pattern="*.pdf"):
        print("Upload one or more PDFs")
        _save(BOOKS_DIR)
    BOOKS = _discover(BOOKS_DIR, "/content", pattern="*.pdf")

if not BOOKS:
    raise SystemExit("no PDF found — re-run this cell and upload at least one")

if GLYPH_MODEL is None:
    # A pipeline ZIP may also be sitting in /content, so the classifier is looked
    # for by name before falling back to any archive at all.
    def _models():
        return _discover(MODEL_DIR, "/content", pattern="*classifier*.zip") or _discover(
            MODEL_DIR, "/content", pattern="*.zip"
        )

    if REUPLOAD or not _models():
        print("Upload chess_glyphs_classifier.zip")
        _save(MODEL_DIR)
    GLYPH_MODEL = next(iter(_models().values()), None)

missing = [label for label, path in BOOKS.items() if not os.path.isfile(path)]
for label in missing:
    print(f"skipping {label}: no file at {BOOKS[label]}")
    del BOOKS[label]
if not BOOKS:
    raise SystemExit("none of the books listed above is on Drive")

width = max(len(label) for label in BOOKS)
for label, path in BOOKS.items():
    first, last = pages_of(label)
    print(f"{label:<{width}} {extract.page_count(path):>5} pages, "
          f"reading {first}-{last}  {path}")
print(f"\nclassifier: {GLYPH_MODEL}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SuperAttaquant            12 pages  /content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/CommentDevenirSuperAttaquant.pdf
BoussoleSurEchiquier      13 pages  /content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/BoussoleSurEchiquier.pdf
ChessStrategy_Grivas_1    10 pages  /content/drive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/ChessTrategy_Grivas_1.pdf

classifier: /content/drive/MyDrive/entrainement_ocr_echecs/chess_glyphs_classifier.zip


## 4. What each book is

Detection runs on a sample of each book. The result is a **conclusion to confirm**,
not a question asked cold: the counts supporting it are printed underneath.

The column that decides the rest of the run is the last one.

- `style` — `figurine_unicode` and `letters` are read straight from the text layer.
- `neutral` — moves naming no piece at all (pawn moves, castlings). Present in every
  language, so it proves "this is chess notation" without favouring one.
- `glyph recovery` — `REQUIRED` means the piece symbols are drawn on the page and
  the text layer never held them. Three kinds of book land here: a scan, a figurine
  font caught by its font name, and the one that fools every other test — a book
  reported as `letters` at 0% confidence with every language scoring zero, whose
  fonts look ordinary (`Helvetica`) and whose only moves are the neutral ones. A
  book of nothing but pawn moves does not exist, so that combination *is* the
  signature of drawn symbols.


In [ ]:
samples, reports = {}, {}
for label, path in BOOKS.items():
    first, last = pages_of(label)
    samples[label] = extract.extract_pages(path, first_page=first, last_page=last)
    reports[label] = notation.detect_notation(samples[label])

width = max(len(label) for label in BOOKS)
header = f"{'book':<{width}} {'style':<17} {'lang':<5} {'conf':>5} {'neutral':>8}  glyph recovery"
print(header)
print("-" * len(header))
for label, report in reports.items():
    print(
        f"{label:<{width}} {report.style:<17} {str(report.language or '-'):<5} "
        f"{report.confidence:>4.0%} {report.neutral_move_count:>8}  "
        f"{'REQUIRED' if report.needs_glyph_recovery else 'not needed'}"
    )


In [ ]:
# The evidence behind each verdict, book by book.
for label, report in reports.items():
    print(f"\n{'=' * 72}\n{label}\n{'=' * 72}")
    print(report.summary())
    print("\nMost frequent fonts:")
    for name, count in list(extract.font_inventory(samples[label]).items())[:5]:
        print(f"  {count:>7}  {name}")


### Read one book's layer with your own eyes

`SELECTED` names the book every single-book check below works on — the previews in
4b, the boxes in step 7, the move tree in step 8. Change it and re-run those cells to
move to another book.

If the moves below show up as garbage — `4)xf7`, `We6`, `2xf7` — the layer is OCR
output. Its prose is usually fine and its move numbers and squares mostly right; only
the piece symbols are hopeless, which is what step 4b is for. Prose that reads cleanly
while moves do not is the signature.


In [ ]:
# The book under investigation. Everything from 4b down reads it; the
# pipeline of section 5 runs on all of them regardless.
SELECTED = "ChessStrategy_Grivas_1"

page = samples[SELECTED][len(samples[SELECTED]) // 2]
print(f"--- {SELECTED}, page {page.number} ({page.width} x {page.height} pt) ---")
print(page.text[:1200])


## 4b. Preview the recovered symbols

Skip this if no book was marked `REQUIRED` above; step 5 applies the classifier on
its own either way. This section exists to look at what it does before trusting it,
on `SELECTED`.

The classifier is a random forest over five classes — King, Queen, Rook, Bishop,
Knight — trained outside this repository. It has no "not a piece" class, so a crop is
only shown to it when its shape already says piece: about twice as wide as the page's
letters, and nearly square. What comes back above `min_confidence` is written into the
pages as a figurine, carrying the box of the printed symbol rather than of whatever
the scanner read there.

Measured on two hand-read pages of a French scanned book, 54 symbols printed: **53
recovered, none invented**, at the default confidence of 0.45. Lower it and phantom
pieces appear in the prose; raise it and bishops start dropping out first.


In [ ]:
# Page 3 of Grivas is where section 10 found the damage: its opening line
# loses every piece symbol. Page 1 recovers all of them and shows nothing.
# None takes the first two pages of whatever range SELECTED is read over,
# which is the only sane default once a book is not read from page 1.
PREVIEW_PAGES = (3, 4)
if PREVIEW_PAGES is None:
    PREVIEW_PAGES = (pages_of(SELECTED)[0], pages_of(SELECTED)[0] + 1)
MIN_CONFIDENCE = None  # None uses glyphs.DEFAULT_MIN_CONFIDENCE

from rce_pipeline import glyphs, scan

classifier = glyphs.GlyphClassifier.load(GLYPH_MODEL)
min_confidence = glyphs.DEFAULT_MIN_CONFIDENCE if MIN_CONFIDENCE is None else MIN_CONFIDENCE

preview = extract.extract_pages(
    BOOKS[SELECTED], first_page=PREVIEW_PAGES[0], last_page=PREVIEW_PAGES[1]
)
# Keyed by the line's position on the page, not by its index: repairing a page
# changes which of its lines carry a move, so pairing by index slides the two
# lists against each other and shows a `before` beside an unrelated `after`.
before = {
    p.number: [
        (line.bbox.y, line.text)
        for line in scan.notation_lines(scan.segment_lines(p))
    ]
    for p in preview
}
repaired, found = glyphs.recover_pieces(
    BOOKS[SELECTED], preview, classifier, min_confidence=min_confidence
)

placed, total = glyphs.placement_score(repaired)
if total:
    print(f"{total} symbols recovered, {placed} spliced into a move ({placed / total:.0%})\n")
else:
    print("no symbols found — check MIN_CONFIDENCE, and that these pages carry moves\n")
for page in repaired:
    for line in scan.notation_lines(scan.segment_lines(page)):
        match = min(
            before[page.number],
            key=lambda entry: abs(entry[0] - line.bbox.y),
            default=None,
        )
        if match is None or abs(match[0] - line.bbox.y) > 3.0:
            continue
        if match[1] != line.text:
            print(f"  before: {match[1]}\n  after : {line.text}\n")


A low "spliced into a move" share means the symbols were recognised and then written
into the wrong characters. That is not the classifier: it is the text layer's boxes.
Tesseract divides a word's box evenly among the characters it read, so a layer that
read `tZJg3` where `♘g3` is printed puts every box in the wrong place, and the symbol
lands beside the move instead of at its head. Around 90% is a well-boxed layer; below
60% the moves from this book are not worth parsing, and the page images would have to
be read in full rather than repaired.


In [ ]:
# The crops the classifier was shown, with what it made of them.
from PIL import Image
import io

page = repaired[0]
source = next(p for p in preview if p.number == page.number)
lines = scan.notation_lines(scan.segment_lines(source))
with scan.PageRenderer(BOOKS[SELECTED]) as renderer:
    for line in lines[:4]:
        image = renderer.crop(line)
        display(Image.open(io.BytesIO(image.png)))
        on_line = [
            g for g in found
            if g.page == page.number
            and line.bbox.y <= g.bbox.y + g.bbox.h / 2 <= line.bbox.y + line.bbox.h
        ]
        print("  ".join(
            f"{g.figurine} {g.confidence:.2f}"
            for g in sorted(on_line, key=lambda g: g.bbox.x)
        ) or "(nothing)")


## 4c. Why a symbol was never offered to the classifier

Four settings were measured and cleared on 2026-08-13 — the confidence threshold, both
shape bounds, and the classifier itself. None of them is what loses the remaining
symbols, which is about one piece move in six, and each loss breaks a whole line.

What is left is the step before all of them. A glyph is only a candidate if its ink
forms **one** connected component of roughly the right size. A rook drawn with its
crenellations detached from its body yields two or three small ones instead, each about
as wide as a letter — so it lands in the letter peak and is never submitted at all.
The text layer shows the same split from the other side, as `l:`, `ll` or `.l:` where a
rook is printed.

That cannot be settled by another statistic; it has to be looked at. This draws every
component the finder saw on a line that still carries rook leftovers, **green where it
was submitted to the classifier and red where it was rejected**.

Read it as one question: is the rook one red box, or several? One red box means the
shape bounds are wrong for this face after all. Several means the ink is fragmented, and
the fix belongs in `_components` — merging neighbours that together make a piece-sized,
nearly-square box — not in any threshold.


In [ ]:
# Run this after 4b, which leaves `preview`, `repaired` and `found` in memory.
from PIL import Image, ImageDraw
import io, re

# A leftover only counts when it sits in front of a square. Matched as a plain
# substring, `ll` also occurs in "usually", which spent a whole run drawing a
# line of prose; the trailing digit is what separates a broken move from an
# English word.
ROOK_LEFTOVERS = re.compile(r"[l.:iJ1t\\']{2,4}[a-h]?x?[a-h][1-8]\b")
LINES = None  # a list of indices from the listing below overrides the search
MAX_LINES = 3

page = repaired[0]
source = next(p for p in preview if p.number == page.number)
lines = scan.notation_lines(scan.segment_lines(source))
# Matched on position with a tolerance, never by index or exact key: repairing
# a page shifts its boxes slightly and changes which lines carry a move, so
# both an index and a rounded key silently pair the wrong two.
repaired_lines = [
    (line.bbox.y, line.text)
    for line in scan.notation_lines(scan.segment_lines(page))
]


def text_after(y):
    if not repaired_lines:
        return ""
    top, text = min(repaired_lines, key=lambda entry: abs(entry[0] - y))
    return text if abs(top - y) <= 3.0 else ""


# The whole page is listed, not just what matched: when the search finds
# nothing, the listing is what says which line to draw by hand.
texts = [text_after(line.bbox.y) for line in lines]
print(f"--- {SELECTED}, page {page.number}, {len(texts)} notation lines ---")
for index, text in enumerate(texts):
    mark = "<<" if ROOK_LEFTOVERS.search(text) else "  "
    print(f"{index:>3} {mark} {text[:96]}")
print()

chosen = LINES if LINES is not None else [
    index for index, text in enumerate(texts) if ROOK_LEFTOVERS.search(text)
]
if not chosen:
    print("no line on this page still carries a broken piece — read the listing\n"
          "above and set LINES to the indices worth drawing.")

with scan.PageRenderer(BOOKS[SELECTED]) as renderer:
    shots = [(line, renderer.crop(line)) for line in lines]

# The reference width is the median over the whole page, exactly as find_glyphs
# does it: a short line has too few letters to supply its own.
boxes_by_line = [glyphs._components(glyphs._to_array(img.png)) for _, img in shots]
reference = glyphs._median_width([b for line in boxes_by_line for b in line]) or 1.0
print(f"a letter is {reference:.1f} px wide\n")

for index in chosen[:MAX_LINES]:
    line, image = shots[index]
    shot = Image.open(io.BytesIO(image.png)).convert("RGB")
    draw = ImageDraw.Draw(shot)
    offered, rejected = [], []
    for (x, y, w, h) in boxes_by_line[index]:
        ratio, aspect = w / reference, w / h
        submitted = (
            glyphs.MIN_WIDTH_RATIO <= ratio <= glyphs.MAX_WIDTH_RATIO
            and aspect <= glyphs.MAX_ASPECT
        )
        draw.rectangle(
            [x, y, x + w, y + h],
            outline=(0, 170, 0) if submitted else (220, 0, 0),
            width=1,
        )
        (offered if submitted else rejected).append(f"{ratio:.2f}w/{aspect:.2f}a")
    display(shot)
    print(f"{index:>3}  {texts[index]}")
    print(f"  green: {'  '.join(offered) or '-'}")
    print(f"  red:   {'  '.join(rejected[:14])}\n")


## 5. Full pipeline, on every book

`FIRST_PAGE` / `LAST_PAGE` restrict the work to part of each book — start small, on a
chapter whose content you know, before launching 400 pages.

- `strict_numbering=True` only reads a move when a move number has just announced one,
  or when variation brackets make the context unambiguous. That is what separates `Bb5`
  from a figure caption reading "diagram b4". Set it to `False` for a book that prints
  long unnumbered sequences.
- `FORCE_LANGUAGE` is worth setting **for any letters book you know**. It decides which
  alphabet piece initials come from, and the alphabets overlap: `R` is the King in
  French and the Rook in English. Both readings are frequently legal in the same
  position, so a wrong language does not fail — it produces a different game. Figurine
  books do not need it: their symbols map to SAN letters directly.
- `glyph_model` is passed for every book and engaged only where step 4 said `REQUIRED`.

| Language | King | Queen | Rook | Bishop | Knight |
| --- | --- | --- | --- | --- | --- |
| `en` | K | Q | R | B | N |
| `fr` | R | D | T | F | C |
| `de` | K | D | T | L | S |
| `es` / `it` | R | D | T | A | C |
| `nl` | K | D | T | L | P |

One book failing does not stop the others: its traceback is printed and the loop moves
on.


In [ ]:
STRICT_NUMBERING = True
FORCE_LANGUAGE = {}       # label -> "fr" / "en" / "de" / "es" / "it" / "nl"
FORCE_NOTATION = {}       # label -> "figurine_unicode" / "letters", to bypass step 2

import traceback

results = {}
for label, path in BOOKS.items():
    print(f"\n{'=' * 72}\n{label}\n{'=' * 72}")
    try:
        results[label] = pipeline.run(
            path,
            work_dir=f"/content/work/{label}",
            output_path=f"/content/{label}.rce",
            first_page=pages_of(label)[0],
            last_page=pages_of(label)[1],
            strict_numbering=STRICT_NUMBERING,
            force_notation=FORCE_NOTATION.get(label),
            force_language=FORCE_LANGUAGE.get(label),
            glyph_model=GLYPH_MODEL,  # engaged only where step 4 said REQUIRED
        )
        print(results[label].report())
    except Exception:
        traceback.print_exc()


## 6. The measurement

### Read `trusted`, not `ok`

`ok` says a legal reading was found. It does not say the position was the book's.
When a move finds no legal reading, the line stays on the position before it and the
score goes on being read there — so every move below a break is played on a board the
book never reached, and the ones that happen to be legal are recorded `ok` at full
confidence anyway. On Sakaev that was 277 of 527 before the numbering was tightened:
more than half the score, counted as sound.

So `broken` is split into the lines that actually died and what merely followed them,
and `ok` into the moves no break stands above and the rest.

| Column | Meaning |
| --- | --- |
| `trusted` | **the figure to compare between two runs** — `ok` moves with no break above them |
| `ok` | every legal reading, sound or not |
| `brk` | every move with no legal reading |
| `breaks` | of those, the lines that died: one per line, and the number worth working on |
| `below` | `ok` moves standing under a break: legal, on a position that is not the book's |

Ten breaks fixed are worth far more than ten moves repaired: each one carries a whole
line back with it.

### The diagrams, where a book prints them as text

A publisher using a **diagram font** writes each position as eight rows of eight
characters, and `diagrams.py` reads them back: the font's letters are learned from the
book itself, wherever a game reached a diagram without breaking. A diagram then either
`confirms` the board the parser had reached, `corrects` it, or `seeds` a game that
opens on a picture and had no starting position at all. Of the corpus, only Sakaev
prints them in the text layer; a book whose diagrams are images shows nothing here.

A high `corrects` is not a complaint about the diagrams — it is the count of places the
score had drifted and nobody could see it. **The moves above a correction are still
counted as `trusted` although the correction proves one of them wrong**; that is the
next thing to fix in the measurement.

### Check the parse is healthy first

The pipeline starts every game from the standard initial position. On a page whose
game begins at move 23, every move is played on a board unrelated to the book,
everything comes out `broken`, and the ambiguity counts below are noise.

So read `trusted` first. If it is near zero, the page range is wrong — change it until
a game starts at `1.` **Do not interpret the ambiguity columns until it looks sane.**

### Then read the ambiguity columns

An ambiguity is a move naming a square two pieces reach — `Nd2` with knights on b1 and
f3 — where the book printed a disambiguating letter (`Nbd2`) the pipeline did not see.
`python-chess` already excludes a pinned piece's moves from `legal_moves`, so the usual
reason a book omits the letter never produces a false ambiguity here. That is what
makes the signal clean, and worth measuring rather than guessing at.

| Column | Meaning |
| --- | --- |
| `amb` | total ambiguous moves |
| `amb-rep` | **the number that answers the question** — cases sitting below a move accepted after an OCR repair, so the board may already be wrong |
| `amb-clean` | cases on lines with no repair above them: the board is sound and the token itself lost the letter |

| Result | Conclusion | Next |
| --- | --- | --- |
| `amb-rep` dominates | the ambiguity is a symptom, not the disease | tighten `_MAX_REPAIR_COST`, and **do not** build the lookahead |
| `amb-clean` dominates | the board is sound, the token lost the letter | build the lookahead |
| `amb` near zero on every book | the problem is theoretical for this corpus | surfacing the candidates to the app is enough |

`nearest_repair_plies` gives the distance from each case to the repair above it: a 1
or 2 damns that repair, a 9 is more likely coincidence.


In [ ]:
from collections import Counter

width = max(len(label) for label in results) if results else 4
header = (
    f"{'book':<{width}} {'moves':>6} {'trusted':>8} {'ok':>6} {'unc':>5} {'brk':>5} "
    f"{'breaks':>7} {'below':>6}   {'amb':>4} {'amb-rep':>8} {'amb-clean':>10} {'figurine':>9}"
)
print(header)
print("-" * len(header))
for label, result in results.items():
    counts = result.parsed.counts()
    breaks = result.parsed.break_diagnosis()
    diagnosis = result.parsed.ambiguity_diagnosis()
    print(
        f"{label:<{width}} {counts['moves']:>6} {breaks['clean']:>8} {counts['ok']:>6} "
        f"{counts['uncertain']:>5} {counts['broken']:>5} "
        f"{breaks['first_breaks']:>7} {breaks['below_break']:>6}   "
        f"{diagnosis['total']:>4} {diagnosis['downstream_of_repair']:>8} "
        f"{diagnosis['clean_line']:>10} {diagnosis['settled_from_consumed']:>9}"
    )

for label, result in results.items():
    print(f"\n{label}: {result.parsed.ambiguity_diagnosis()}")

# The positions a book prints as pictures: what each diagram did to the line
# it was printed on. Only books setting diagrams in a diagram font have any.
for label, result in results.items():
    if not result.diagrams:
        continue
    verdicts = Counter(check["verdict"] for check in result.parsed.diagram_checks)
    print(f"\n{label}: {len(result.diagrams)} diagrams read from the text layer — "
          + ", ".join(f"{n} {verdict}" for verdict, n in verdicts.most_common()))


## 7. What did not get through

`broken` first: no legal reading was found, so the move is a hole in the line. Then
`uncertain`, accepted after repairing a look-alike scanning error (`0`/`O`, `1`/`l`,
`8`/`B`) — the `repair` field says what was substituted.

Repairs are deliberately conservative. Allowing one arbitrary wrong character would
recover more moves and would also turn `Qh9` into `Qh5` and `Nc6` into `Nc3`: squares
differ by a single character all the time, so the pipeline would emit legal but wrong
moves that silently corrupt every position further down the line.

These moves keep their page and their box, so they stay clickable in the app — which
is where they are meant to be corrected.


In [ ]:
result = results[SELECTED]

for move in result.problems(limit=25):
    detail = move.repair["reason"] if move.repair else ""
    print(f"[{move.status:>9}] p.{move.page:>3}  {move.san:<8} conf={move.confidence:.2f}  {detail}")

print(f"\n{len(result.parsed.skipped)} tokens dropped before validation:")
for skipped in result.parsed.skipped[:15]:
    print(f"  p.{skipped['page']:>3}  {skipped['text']:<10} {skipped['reason']}")


## 8. Check the boxes by eye

This is the most useful check in the notebook. A box off by a few points shows up in no
counter, but makes the clickable zone useless in the app. So render the page and draw
the boxes on top of it.

The code converts `.rce` coordinates (origin bottom-left) back to MuPDF's (origin
top-left) — the same round trip Flutter makes, in reverse. If the frames land on the
moves, the convention is right on both sides.


In [ ]:
try:
    import pymupdf as fitz
except ImportError:
    import fitz
from PIL import Image, ImageDraw

result = results[SELECTED]
PREVIEW_PAGE = result.parsed.moves[0].page if result.parsed.moves else pages_of(SELECTED)[0]
ZOOM = 2.0
STATUS_COLOURS = {"ok": (0, 160, 0), "uncertain": (220, 140, 0), "broken": (210, 0, 0)}

doc = fitz.open(BOOKS[SELECTED])
page = doc[PREVIEW_PAGE - 1]
pixmap = page.get_pixmap(matrix=fitz.Matrix(ZOOM, ZOOM))
image = Image.frombytes("RGB", (pixmap.width, pixmap.height), pixmap.samples)
draw = ImageDraw.Draw(image)

page_height = page.rect.height
drawn = 0
for move in result.parsed.moves:
    if move.page != PREVIEW_PAGE:
        continue
    b = move.bbox
    top = page_height - b.y - b.h  # flip back to MuPDF's top-left origin
    draw.rectangle(
        [b.x * ZOOM, top * ZOOM, (b.x + b.w) * ZOOM, (top + b.h) * ZOOM],
        outline=STATUS_COLOURS[move.status],
        width=2,
    )
    drawn += 1

doc.close()
print(f"{SELECTED}: {drawn} boxes drawn on page {PREVIEW_PAGE}")
image


## 9. Check the move tree

Variations are reconstructed from `parent_id`, never from array order. This display
follows those links, which checks along the way that they are coherent.


In [ ]:
from collections import defaultdict

GAME_INDEX = 0     # which game to show
MAX_LINES = 80

result = results[SELECTED]
game = result.parsed.games[GAME_INDEX]
children = defaultdict(list)
for move in result.parsed.moves:
    if move.game_id == game.id:
        children[move.parent_id].append(move)
for siblings in children.values():
    siblings.sort(key=lambda m: m.variation_index)

printed = 0

def show(move_id, depth):
    global printed
    for child in children[move_id]:
        if printed >= MAX_LINES:
            return
        printed += 1
        number = f"{(child.ply + 1) // 2}." + ("" if child.ply % 2 else "..")
        mark = {"ok": " ", "uncertain": "~", "broken": "!"}[child.status]
        note = f"    [{child.comment[:60]}]" if child.comment else ""
        print("  " * depth + f"{mark} {number}{child.san}{note}")
        # Only a variation shifts the indentation; the main line stays flush.
        show(child.id, depth + 1 if child.variation_index else depth)

title = game.title or "(untitled)"
print(f"=== {SELECTED} / {game.id} - {title} (p.{game.page_start}) ===")
show(None, 0)


## 10. Audit one page

Step 8 draws boxes and step 9 walks the tree, but neither answers the question a
missing box actually raises: *this move is printed on the page, so where did it go?*
A move can be lost at several places, and the fix differs at each.

This lists every move-shaped string in the page's text layer and what became of it:

- **`ok` / `uncertain` / `broken`** — it became a move. `broken` means no legal
  reading was found, which on a book whose games start mid-analysis says nothing
  about how well the move was read.
- **`skipped`** — it became a move token and the parser refused it, with the reason
  printed. `no move number in context` is `strict_numbering` declining a move no
  number announced: what happens to moves quoted inline in prose, and to a square
  named in a figure caption.
- **`token:text`** — the tokeniser read that span as prose, not as a move.
- **`NO TOKEN`** — nothing was emitted there at all. On a book needing glyph
  recovery this is the one that points upstream: usually the symbol was never
  spliced into the move, leaving a fragment nothing recognises.

`broken` is a legality problem and `NO TOKEN` is a reading problem; they are fixed in
opposite places, which is the whole reason for separating them here.

**One blind spot, deliberate.** A move whose square was misread into something that
is not a square — `Qh9`, `Bi4` — matches nothing and appears on no line below. The
header counts are the cross-check: if the tokens and moves do not roughly account for
the lines listed, the difference is made of those.


In [ ]:
AUDIT_BOOK = SELECTED
AUDIT_PAGE = None  # None picks the page carrying the most moves

import re
from collections import Counter

result = results[AUDIT_BOOK]

if AUDIT_PAGE is None:
    seen = Counter(m.page for m in result.parsed.moves)
    AUDIT_PAGE = seen.most_common(1)[0][0] if seen else pages_of(AUDIT_BOOK)[0]

page = next((p for p in result.pages if p.number == AUDIT_PAGE), None)
if page is None:
    raise SystemExit(f"page {AUDIT_PAGE} is outside the range that was run")

# Anything shaped like a move, figurine or letter, so that a move the tokeniser
# never saw still turns up here. Deliberately looser than the tokeniser's own
# pattern: the point is to catch what it missed.
MOVE_LIKE = re.compile(
    r"(?<![A-Za-z0-9])"
    r"(?:O-O(?:-O)?|[\u2654-\u265FKQRBN]?[a-h]?[1-8]?x?[a-h][1-8](?:=[QRBN])?)"
    r"[+#!?]*"
)

tokens = [t for t in result.tokens if t.page == AUDIT_PAGE]
moves = [m for m in result.parsed.moves if m.page == AUDIT_PAGE]
skipped = [s for s in result.parsed.skipped if s["page"] == AUDIT_PAGE]
recovered = [g for g in result.glyphs if g.page == AUDIT_PAGE]

# Skipped entries carry no offsets, so they are matched back to their token by
# printed text. Two identical moves on one page are told apart by order only.
pending = {}
for entry in skipped:
    pending.setdefault(entry["raw"], []).append(entry["reason"])
status_of = {id(m): m.status for m in moves}
move_queue = list(moves)

print(f"=== {AUDIT_BOOK} — page {AUDIT_PAGE} ===")
print(f"{len(recovered)} symbols recovered   {len(tokens)} tokens   "
      f"{len(moves)} moves   {len(skipped)} skipped\n")

covering = {}
for token in tokens:
    for offset in range(token.start, token.end):
        covering[offset] = token

verdicts = Counter()
rows = []
for match in MOVE_LIKE.finditer(page.text):
    token = covering.get(match.start())
    if token is None or token.kind != "move":
        verdict = "NO TOKEN" if token is None else f"token:{token.kind}"
        detail = ""
    elif pending.get(token.raw):
        verdict, detail = "skipped", pending[token.raw].pop(0)
    else:
        move = move_queue.pop(0) if move_queue else None
        verdict = move.status if move else "?"
        detail = (move.repair or {}).get("reason", "") if move else ""
    verdicts[verdict] += 1
    rows.append((match.group(), verdict, detail))

for printed, verdict, detail in rows:
    print(f"  {printed:<12} {verdict:<10} {detail[:60]}")

print("\n" + "  ".join(f"{v}={n}" for v, n in verdicts.most_common()))
if verdicts["NO TOKEN"]:
    print(f"\n{verdicts['NO TOKEN']} printed moves produced no token at all — "
          "a reading problem, upstream of legality.")


## 11. Download the archives

Each `.rce` holds one original PDF **unchanged**, plus `manifest.json` and
`moves.json`. These are the files the Flutter app imports.


In [ ]:
import zipfile
from google.colab import files

for label, result in results.items():
    if not result.rce_path:
        continue
    print(f"\n{label} — {result.rce_path}")
    with zipfile.ZipFile(result.rce_path) as archive:
        for info in archive.infolist():
            print(f"  {info.file_size:>12,} B  {info.filename}")

for result in results.values():
    if result.rce_path:
        files.download(result.rce_path)
